# Qwen3-ASR-0.6B 中文鲁棒后训练全链路编排（Phase 0 → Phase 5）

本 notebook 面向普通内存单张 A100 Colab runtime，仅负责按顺序编排 CLI 调用（`Phase 0 Baseline` → `Phase 1 Manifest` → `Phase 2 SFT` → `Phase 3 DPO` → `Phase 4 RL` → `Phase 5 Sealed test.jsonl 四阶段终局评测`）。所有模型加载、训练、门禁、release 读回和评测逻辑均位于 `scripts/`、`train/`、`inference/`、`evaluation/` 模块与 `configs/` 配置文件中，不在单元格内保存隐藏状态。


In [ ]:
%cd /content/lora-asr
!python3 scripts/rebuild_phase0_env.py \
  --lock requirements.lock \
  --output-dir /content/run/phase0_env_rebuild
!python3 scripts/run_phase0_baseline.py \
  --config configs/inference/base_eval.yaml \
  --output-dir /content/run/phase0_baseline \
  --run-id phase0_baseline \
  --seed 42 \
  --drive-root /content/drive/MyDrive/qwen3-asr


## Phase 1：五类退化场景数据校验


In [ ]:
!python3 scripts/manifest_validate.py \
  --manifest /content/cache/aishell_pilot_20261015/manifest \
  --config configs/data/manifest_mvp.yaml \
  --check-audio


## Phase 2：SFT 训练与不可变 Release 读回发布


In [ ]:
!QWEN3_ASR_DISABLE_RUNTIME_BUDGET=1 python3 train/train_sft.py \
  --config configs/train/sft_pilot_lr1e5.yaml \
  --manifest /content/cache/aishell_pilot_20261015/manifest/train.jsonl \
  --val-manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl \
  --output-dir /content/run/20261015T234939_sft_06b_zh_seed42 \
  --run-id 20261015T234939_sft_06b_zh_seed42 \
  --seed 42
!python3 train/sft_release.py prepare \
  --config configs/train/sft_pilot_lr1e5.yaml \
  --run-dir /content/run/20261015T234939_sft_06b_zh_seed42 \
  --output-dir /content/drive/MyDrive/qwen3-asr/releases/sft/sft_06b_zh_20261015T234939_step000030 \
  --release-id sft_06b_zh_20261015T234939_step000030
!python3 train/sft_release.py readback \
  --config configs/train/sft_pilot_lr1e5.yaml \
  --release-dir /content/drive/MyDrive/qwen3-asr/releases/sft/sft_06b_zh_20261015T234939_step000030 \
  --manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl


## Phase 3：DPO 对齐单字替换 + Dropout 父假设锚训练与 Release 读回发布


In [ ]:
!QWEN3_ASR_DISABLE_RUNTIME_BUDGET=1 python3 train/train_dpo.py \
  --config configs/train/dpo_pilot_edit1_anchor_step40_h100.yaml \
  --manifest /content/cache/aishell_pilot_20261015/manifest/train.jsonl \
  --val-manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl \
  --pair-train /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/ref_logprobs/pairs_logprob/train_pairs.jsonl \
  --pair-val /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/ref_logprobs/pairs_logprob/val_pairs.jsonl \
  --reference-cache /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/ref_logprobs/ref_logprobs.jsonl \
  --reference-report /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/ref_logprobs/ref_report.json \
  --pair-metadata /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/ref_logprobs/pairs_logprob/pair_metadata.json \
  --release-dir /content/drive/MyDrive/qwen3-asr/releases/sft/sft_06b_zh_20261015T234939_step000030 \
  --audio-root /content/cache/aishell_pilot_20261015/manifest \
  --null-pairs-dir /content/drive/MyDrive/qwen3-asr/dpo/sft_06b_zh_20261015T234939_step000030/pairs \
  --output-dir /content/run/20261022T133007_dpo_06b_zh_seed42 \
  --run-id 20261022T133007_dpo_06b_zh_seed42 \
  --seed 42
!python3 train/dpo_release.py prepare \
  --config configs/train/dpo_pilot_edit1_anchor_step40_h100.yaml \
  --run-dir /content/run/20261022T133007_dpo_06b_zh_seed42 \
  --output-dir /content/run/releases/dpo/dpo_06b_zh_20261022T133007_step000040 \
  --release-id dpo_06b_zh_20261022T133007_step000040 \
  --sync-dir /content/drive/MyDrive/qwen3-asr/releases/dpo/dpo_06b_zh_20261022T133007_step000040
!python3 train/dpo_release.py readback \
  --config configs/train/dpo_pilot_edit1_anchor_step40_h100.yaml \
  --release-dir /content/run/releases/dpo/dpo_06b_zh_20261022T133007_step000040 \
  --manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl \
  --sync-dir /content/drive/MyDrive/qwen3-asr/releases/dpo/dpo_06b_zh_20261022T133007_step000040


## Phase 4：RL（`group_pg_mixed_v1` 对齐差异掩码 + Reference Freeze）训练与 Release 读回发布


In [ ]:
!QWEN3_ASR_DISABLE_RUNTIME_BUDGET=1 python3 train/train_rl.py \
  --config configs/train/rl_pilot_aligned.yaml \
  --manifest /content/cache/aishell_pilot_20261015/manifest/train.jsonl \
  --val-manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl \
  --release-dir /content/drive/MyDrive/qwen3-asr/releases/dpo/dpo_06b_zh_20261022T133007_step000040 \
  --audio-root /content/cache/aishell_pilot_20261015/manifest \
  --output-dir /content/run/20261023T042034_rl_06b_zh_seed42 \
  --run-id 20261023T042034_rl_06b_zh_seed42 \
  --seed 42
!python3 train/rl_release.py prepare \
  --config configs/train/rl_pilot_aligned.yaml \
  --run-dir /content/run/20261023T042034_rl_06b_zh_seed42 \
  --output-dir /content/run/releases/rl/rl_06b_zh_20261023T042034_step000005 \
  --release-id rl_06b_zh_20261023T042034_step000005 \
  --sync-dir /content/drive/MyDrive/qwen3-asr/releases/rl/rl_06b_zh_20261023T042034_step000005
!python3 train/rl_release.py readback \
  --config configs/train/rl_pilot_aligned.yaml \
  --release-dir /content/run/releases/rl/rl_06b_zh_20261023T042034_step000005 \
  --manifest /content/cache/aishell_pilot_20261015/manifest/val.jsonl \
  --sync-dir /content/drive/MyDrive/qwen3-asr/releases/rl/rl_06b_zh_20261023T042034_step000005


## Phase 5：封存 2500 条 `test.jsonl` 四阶段（Base / SFT / DPO / RL）同口径终局评测


In [ ]:
!python3 scripts/run_phase5_test_benchmark.py \
  --config configs/train/rl_pilot_aligned.yaml \
  --test-manifest /content/cache/aishell_pilot_20261015/manifest/test.jsonl \
  --manifest-sha-index /content/cache/aishell_pilot_20261015/manifest/sha256.json \
  --data-config configs/data/manifest_mvp.yaml \
  --pilot-sample-tar /content/drive/MyDrive/qwen3-asr/data/aishell1/20261015/pilot_sample_audio.tar \
  --sft-release-dir /content/drive/MyDrive/qwen3-asr/releases/sft/sft_06b_zh_20261015T234939_step000030 \
  --dpo-release-dir /content/drive/MyDrive/qwen3-asr/releases/dpo/dpo_06b_zh_20261022T133007_step000040 \
  --rl-release-dir /content/drive/MyDrive/qwen3-asr/releases/rl/rl_06b_zh_20261023T042034_step000005 \
  --output-dir /content/run/20261023_phase5_test_06b_zh_seed42 \
  --run-id 20261023_phase5_test_06b_zh_seed42 \
  --seed 42 \
  --drive-root /content/drive/MyDrive/qwen3-asr
